# Week 8: Tool Calling & Function Calling

**Lab companion to [week_8.md](../agenda/week_08.md)**

In this lab, you will:
1. Define tools with JSON schemas
2. Handle tool calls from the LLM
3. Build multi-tool agents
4. Implement tool execution with error handling

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. Defining Tools

In [ ]:
# Define tools using JSON schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country, e.g., 'London, UK'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a mathematical calculation",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression, e.g., '2 + 2 * 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("Defined 2 tools:")
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description']}")

## 2. Implementing Tool Functions

In [ ]:
# Implement the actual tool functions
def get_weather(location: str, unit: str = "celsius") -> dict:
    """Mock weather API."""
    # In production: call actual weather API
    mock_data = {
        "london": {"temp": 15, "condition": "cloudy"},
        "tokyo": {"temp": 22, "condition": "sunny"},
        "new york": {"temp": 18, "condition": "partly cloudy"}
    }

    city = location.lower().split(",")[0]
    data = mock_data.get(city, {"temp": 20, "condition": "unknown"})

    temp = data["temp"]
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32

    return {
        "location": location,
        "temperature": temp,
        "unit": unit,
        "condition": data["condition"]
    }

def calculate(expression: str) -> dict:
    """Evaluate math expression safely."""
    import ast
    import operator

    # Safe operators
    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow
    }

    def eval_node(node):
        if isinstance(node, ast.Constant):
            return node.value
        elif isinstance(node, ast.BinOp):
            left = eval_node(node.left)
            right = eval_node(node.right)
            op = operators.get(type(node.op))
            if op:
                return op(left, right)
        raise ValueError(f"Unsupported expression")

    try:
        tree = ast.parse(expression, mode='eval')
        result = eval_node(tree.body)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "error": str(e)}

# Test functions
print(get_weather("London, UK"))
print(calculate("2 + 3 * 4"))

## 3. Tool Calling Flow

In [ ]:
# Map tool names to functions
tool_functions = {
    "get_weather": get_weather,
    "calculate": calculate
}

def call_with_tools(user_message: str) -> str:
    """Complete tool calling flow."""

    messages = [{"role": "user", "content": user_message}]

    # Step 1: Send to LLM with tools
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"  # Let model decide
    )

    message = response.choices[0].message

    # Step 2: If no tool calls, return response
    if not message.tool_calls:
        return message.content

    # Step 3: Execute tool calls
    messages.append(message)  # Add assistant message with tool calls

    for tool_call in message.tool_calls:
        print(f"  🔧 Calling: {tool_call.function.name}({tool_call.function.arguments})")

        # Parse arguments
        args = json.loads(tool_call.function.arguments)

        # Execute function
        func = tool_functions[tool_call.function.name]
        result = func(**args)

        # Add tool result
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    # Step 4: Get final response
    final_response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages
    )

    return final_response.choices[0].message.content

# Test
print("User: What's the weather in Tokyo?")
response = call_with_tools("What's the weather in Tokyo?")
print(f"Assistant: {response}")

In [ ]:
# Test with calculation
print("User: What's 25 * 4 + 100?")
response = call_with_tools("What's 25 * 4 + 100?")
print(f"Assistant: {response}")

In [ ]:
# Test with multiple tools
print("User: Compare the temperatures in London and Tokyo in Fahrenheit")
response = call_with_tools("Compare the temperatures in London and Tokyo in Fahrenheit")
print(f"Assistant: {response}")

## 4. Building a Tool Agent

In [ ]:
class ToolAgent:
    """Agent that can use tools to answer questions."""

    def __init__(self):
        self.client = OpenAI()
        self.tools = tools
        self.tool_functions = tool_functions
        self.messages = []

        self.system_prompt = """You are a helpful assistant with access to tools.
Use tools when needed to get accurate information.
If you don't need a tool, respond directly."""

    def chat(self, user_message: str, max_iterations: int = 5) -> str:
        """Chat with tool support."""

        self.messages.append({"role": "user", "content": user_message})

        for i in range(max_iterations):
            response = self.client.chat.completions.create(
                model="gpt-5-mini",
                messages=[{"role": "system", "content": self.system_prompt}] + self.messages,
                tools=self.tools,
                tool_choice="auto"
            )

            message = response.choices[0].message

            # No tool calls - we have a final answer
            if not message.tool_calls:
                self.messages.append({"role": "assistant", "content": message.content})
                return message.content

            # Execute tools
            self.messages.append(message)

            for tool_call in message.tool_calls:
                result = self._execute_tool(tool_call)
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

        return "Maximum iterations reached"

    def _execute_tool(self, tool_call) -> str:
        """Execute a tool and return result."""
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"  🔧 {name}({args})")

        try:
            func = self.tool_functions[name]
            result = func(**args)
            return json.dumps(result)
        except Exception as e:
            return json.dumps({"error": str(e)})

    def reset(self):
        """Clear conversation history."""
        self.messages = []

# Test agent
agent = ToolAgent()

print("Agent demo:")
print("-" * 40)

questions = [
    "Hi, how are you?",
    "What's the weather in London?",
    "Is it warmer in Tokyo?"
]

for q in questions:
    print(f"\nUser: {q}")
    response = agent.chat(q)
    print(f"Agent: {response}")

## 5. Adding More Tools

In [ ]:
# Add more tools
additional_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to someone",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "Recipient email"},
                    "subject": {"type": "string", "description": "Email subject"},
                    "body": {"type": "string", "description": "Email body"}
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

# Mock implementations
def search_web(query: str) -> dict:
    return {
        "query": query,
        "results": [
            {"title": f"Result 1 for {query}", "snippet": "This is a sample result..."},
            {"title": f"Result 2 for {query}", "snippet": "Another relevant result..."}
        ]
    }

def send_email(to: str, subject: str, body: str) -> dict:
    # In production: actually send email
    return {
        "status": "sent",
        "to": to,
        "subject": subject,
        "message": "Email sent successfully"
    }

# Add to agent
agent.tools.extend(additional_tools)
agent.tool_functions["search_web"] = search_web
agent.tool_functions["send_email"] = send_email

print(f"Agent now has {len(agent.tools)} tools")

In [ ]:
# Test new tools
agent.reset()

print("Testing new tools:")
print("-" * 40)

print("\nUser: Search for Python tutorials")
response = agent.chat("Search for Python tutorials")
print(f"Agent: {response}")

## Summary

You learned:
- ✅ Define tools with JSON schema
- ✅ Implement tool functions
- ✅ Handle tool call responses
- ✅ Build multi-tool agents
- ✅ Error handling in tool execution

**Next:** [Week 11 - Frameworks](week_09_frameworks.ipynb)